# MICE Algorithm - Multivariate Imputation by Chained Equations

- There's three ways that data can be missing - MCAR, MAR, MNAR
- We use MICE when we are sure that the data is MAR, which basically means that the cells value depends on the other columns in the dataset.
- It's very accurate but also slow & you have to deploy the entire training set on the server

# How MICE works?

**1. The Setup (The Quick Fix)**
- The algorithm looks at your dataset full of holes (missing values).
- It slaps a basic placeholder into all of them (usually just the mean or median of each column). Now your dataset is technically "full," even if the guesses are pretty basic.

**2. Target the First Column**
- MICE picks one column to focus on first (let's call it Column A). 
- It removes those basic mean placeholders we just put in for Column A, turning them back to "missing."

**3. Train a Model**
- It treats Column A as the "target" variable to predict. 
- It uses all the *other* columns as features and trains a machine learning model (like Linear Regression or Random Forest) using the rows where Column A already had real, original data.

**4. Predict and Fill**
- It uses that newly trained model to predict what the missing values in Column A should actually be. 
- Boom—it fills them in. Now Column A has way smarter, model-backed guesses instead of just the boring mean.

**5. Chain It (Move to the Next Column)**
- MICE moves to the next column (Column B) and rips out its basic placeholders.
- It trains a new model using all the other columns. **Crucial detail:** the data it uses to train this time *includes* those fresh, smart guesses we just put into Column A! 
- It predicts the missing values for Column B, fills them in, and repeats this process for every single column that has missing data.

**6. Loop It Until It Settles (Convergence)**
- Going through all the columns once is just "Iteration 1." MICE will repeat this entire cycle multiple times (usually around 10 to 20 times).
- **Why?** Because every time it cycles through, the data being used to train the models is getting slightly better and more accurate.
- Eventually, the predictions stabilize and stop changing much from one cycle to the next (this is called **convergence**). Once it hits that point, the algorithm stops, and you are left with your final, heavily-optimized imputed dataset!

In [37]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression

In [38]:
df = np.round(pd.read_csv('50_Startups.csv')[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)

# Randomly punch holes in 10% of the data to simulate missing values
np.random.seed(42)
df_missing = df.copy()
for col in df_missing.columns:
    # Pick 10% of random rows in each column and set them to nan
    missing_indices = df_missing.sample(frac=0.1).index
    df_missing.loc[missing_indices, col] = np.nan

In [39]:
df

,R&D Spend,Administration,Marketing Spend,Profit
0,17.0,14.0,47.0,19.0
1,16.0,15.0,44.0,19.0
2,15.0,10.0,41.0,19.0
3,14.0,12.0,38.0,18.0
4,14.0,9.0,37.0,17.0
5,13.0,10.0,36.0,16.0
6,13.0,15.0,13.0,16.0
7,13.0,15.0,32.0,16.0
8,12.0,15.0,31.0,15.0
9,12.0,11.0,30.0,15.0


In [51]:
missing_coords = []
for col in df_missing.columns:
    for row in df_missing.index:
        if pd.isna(df_missing.loc[row, col]):
            missing_coords.append((row, col))

In [52]:
print(missing_coords)

[(13, 'R&D Spend'), (17, 'R&D Spend'), (30, 'R&D Spend'), (39, 'R&D Spend'), (45, 'R&D Spend'), (2, 'Administration'), (29, 'Administration'), (38, 'Administration'), (45, 'Administration'), (46, 'Administration'), (1, 'Marketing Spend'), (5, 'Marketing Spend'), (29, 'Marketing Spend'), (43, 'Marketing Spend'), (49, 'Marketing Spend'), (13, 'Profit'), (16, 'Profit'), (20, 'Profit'), (40, 'Profit'), (49, 'Profit')]


# Iteration 0 - mean imputation

In [53]:
df_imputed = df_missing.copy()
for col in df_imputed.columns:
    df_imputed[col] = df_imputed[col].fillna(df_imputed[col].mean())

# Loop through MICE to find optimal value that's missing

In [55]:
iterations = 10 

for i in range(iterations):
    
    # Loop through every single missing value we found earlier
    for row, col in missing_coords:
        
        # 1. Sabotage: Temporarily remove the imputed value
        df_imputed.loc[row, col] = np.nan
        
        # 2. Separate features (X) and target (y)
        X = df_imputed.drop(columns=[col])
        y = df_imputed[col]
        
        # 3. Drop the row containing the NaN from our training data
        X_train = X.drop(index=row)
        y_train = y.drop(index=row)
        
        # 4. Train the model on the remaining complete rows
        lr = LinearRegression()
        lr.fit(X_train, y_train)
        
        # 5. Predict the missing value using the other columns
        X_test = X.loc[[row]] 
        prediction = lr.predict(X_test)
        
        # 6. Plug it back in
        df_imputed.loc[row, col] = prediction[0]

In [56]:
print("Original Data (First 5 rows):")
print(df.head())
print("\nImputed Data (First 5 rows):")
print(df_imputed.head())

Original Data (First 5 rows):
   R&D Spend  Administration  Marketing Spend  Profit
0       17.0            14.0             47.0    19.0
1       16.0            15.0             44.0    19.0
2       15.0            10.0             41.0    19.0
3       14.0            12.0             38.0    18.0
4       14.0             9.0             37.0    17.0

Imputed Data (First 5 rows):
   R&D Spend  Administration  Marketing Spend  Profit
0       17.0       14.000000         47.00000    19.0
1       16.0       15.000000         37.15769    19.0
2       15.0       12.558048         41.00000    19.0
3       14.0       12.000000         38.00000    18.0
4       14.0        9.000000         37.00000    17.0
